## Installing the required libraries

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl
!pip install -q rasterio tifffile albumentations
!pip install -q huggingface_hub
!pip install datasets==2.19.2 -q

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Testing and validating the model

In [ ]:
!pip install rouge_score -q
!pip install sentence-transformers -q

In [ ]:
import os
import gc
import json
import re
import random
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
from kaggle_secrets import UserSecretsClient

# ==============================================================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# ==============================================================================
print("\n" + "="*50)
print("INITIALIZING ENVIRONMENT")
print("="*50)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Purge VRAM
for var_name in ['model', 'processor', 'inputs']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
torch.cuda.empty_cache()

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

# ==============================================================================
# 2. LOAD FINE-TUNED MODEL (4-BIT)
# ==============================================================================
# Replace with your actual repository name
MODEL_ID = "nillohitroy/satquery-llava-multitask"
print(f"\nLoading optimized model for inference: {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    torch_dtype=torch.float16,
    token=hf_token,
    attn_implementation="eager"
)
model.eval()

# ==============================================================================
# 3. BUILD BENCHMARK TEST SET (Unseen Data)
# ==============================================================================
print("\n" + "="*50)
print("ASSEMBLING BENCHMARK DATASET")
print("="*50)
test_data = []
TARGET_EVAL_SAMPLES = 150 # 150 per dataset for robust scoring

# --- A. VRSBench Validation Loading ---
print("\nLoading VRSBench validation samples...")
BASE_INPUT_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
if not os.path.exists(BASE_INPUT_DIR):
    BASE_INPUT_DIR = "/kaggle/input/vrs-bench/vrsbench_data"

vrs_img_dir = os.path.join(BASE_INPUT_DIR, "Images_val", "Images_val")
vrs_anno_dir = os.path.join(BASE_INPUT_DIR, "Annotations_val", "Annotations_val")

vrs_loaded = 0
if os.path.exists(vrs_anno_dir) and os.path.exists(vrs_img_dir):
    anno_files = [f for f in os.listdir(vrs_anno_dir) if f.endswith('.json')]
    random.seed(42)
    random.shuffle(anno_files)
    
    for file in anno_files:
        if vrs_loaded >= TARGET_EVAL_SAMPLES: break
            
        with open(os.path.join(vrs_anno_dir, file), 'r') as f:
            data = json.load(f)
        if isinstance(data, dict): data = [data]
            
        for item in data:
            if vrs_loaded >= TARGET_EVAL_SAMPLES: break
            
            img_name = item.get('source') or item.get('image') or file.replace('.json', '.jpg')
            img_path = os.path.join(vrs_img_dir, img_name)
            
            if not os.path.exists(img_path): continue
            
            qa_list = item.get('qa_pairs', [])
            if not qa_list and item.get('caption'):
                qa_list = [{'question': 'Describe this image.', 'answer': item.get('caption')}]
                
            for qa in qa_list:
                if vrs_loaded >= TARGET_EVAL_SAMPLES: break
                test_data.append({
                    'dataset': 'VRSBench',
                    'image_path': img_path,
                    'question': qa.get('question', ''),
                    'answer': qa.get('answer', '')
                })
                vrs_loaded += 1
    print(f"Loaded {vrs_loaded} VRSBench validation samples.")
else:
    print("Warning: VRSBench validation directory not found.")

# --- B. BigEarthNet Local Test Loading ---
print("\nMapping BIFOLD Test Split Annotations...")
ben_test_anno_map = {}
try:
    ben_text_ds = load_dataset("BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", split="all_data", streaming=True)
    for row in ben_text_ds:
        if row.get('split') == 'test':
            patch_id = row.get('patch_id') or row.get('name') or row.get('id')
            if patch_id: ben_test_anno_map[str(patch_id).strip()] = row
except Exception as e:
    print(f"Warning: Could not load BIFOLD test metadata: {e}")

LOCAL_DATA_DIR = "/kaggle/input/datasets/immulu"
valid_test_patches = []
for root, dirs, files in os.walk(LOCAL_DATA_DIR):
    patch_id = os.path.basename(root).strip()
    if patch_id in ben_test_anno_map:
        valid_test_patches.append((patch_id, root))

sampled_test_patches = random.sample(valid_test_patches, min(TARGET_EVAL_SAMPLES, len(valid_test_patches)))

print(f"Compositing {len(sampled_test_patches)} BigEarthNet test images...")
ben_loaded = 0
for patch_id, patch_dir in sampled_test_patches:
    anno = ben_test_anno_map[patch_id]
    question = anno.get("input", "What land cover classes are present in this satellite image?")
    answer = anno.get("output", "")
    
    try:
        b04_path = os.path.join(patch_dir, [f for f in os.listdir(patch_dir) if f.endswith('B04.tif')][0])
        b03_path = os.path.join(patch_dir, [f for f in os.listdir(patch_dir) if f.endswith('B03.tif')][0])
        b02_path = os.path.join(patch_dir, [f for f in os.listdir(patch_dir) if f.endswith('B02.tif')][0])
        
        r, g, b = np.array(Image.open(b04_path)), np.array(Image.open(b03_path)), np.array(Image.open(b02_path))
        rgb = np.stack([r, g, b], axis=-1)
        rgb_min, rgb_max = rgb.min(), rgb.max()
        if rgb_max > rgb_min:
            rgb = ((rgb - rgb_min) / (rgb_max - rgb_min) * 255.0)
            
        img = Image.fromarray(rgb.astype(np.uint8)).convert('RGB')
        
        if answer:
            test_data.append({
                'dataset': 'BigEarthNet',
                'image_obj': img,
                'question': question,
                'answer': answer
            })
            ben_loaded += 1
    except Exception:
        continue

print(f"Loaded {ben_loaded} BigEarthNet test samples.")
print(f"\nTotal benchmark dataset assembled: {len(test_data)} samples")
random.shuffle(test_data)

# ==============================================================================
# 4. BENCHMARK EXECUTION
# ==============================================================================
print("\n" + "="*50)
print("EXECUTING INFERENCE & SCORING")
print("="*50)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
semantic_scorer = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

SIMILARITY_THRESHOLD = 0.55
vrsbench_scores = []
bigearth_semantic_precision = []
bigearth_semantic_recall = []

def clean_labels(text):
    text = re.sub(r"^(the (land cover )?features present are:?|this image (shows|contains):?)", "", text, flags=re.IGNORECASE)
    text = text.replace(".", "").strip()
    return [item.strip().lower() for item in re.split(r"[,;\n]", text) if item.strip()]

for sample in tqdm(test_data, desc="Evaluating"):
    if sample['dataset'] == 'VRSBench':
        try:
            test_image = Image.open(sample['image_path']).convert("RGB")
        except: continue
    else:
        test_image = sample['image_obj']

    prompt = f"USER: <image>\n{sample['question']}\nASSISTANT:"
    
    try:
        inputs = processor(text=prompt, images=test_image, return_tensors="pt").to("cuda", torch.float16)
        
        with torch.inference_mode():
            generated_ids = model.generate(**inputs, max_new_tokens=80, do_sample=False)
            
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        prediction = generated_text.split("ASSISTANT:")[-1].strip()
        ground_truth = sample['answer'].strip()

        if sample['dataset'] == "VRSBench":
            score = scorer.score(ground_truth, prediction)
            vrsbench_scores.append(score['rougeL'].fmeasure)

        elif sample['dataset'] == "BigEarthNet":
            pred_list = clean_labels(prediction)
            gt_list = clean_labels(ground_truth)

            if not pred_list or not gt_list:
                if not pred_list and gt_list:
                    bigearth_semantic_precision.append(0.0)
                    bigearth_semantic_recall.append(0.0)
                continue

            pred_emb = semantic_scorer.encode(pred_list, convert_to_tensor=True, device='cpu')
            gt_emb = semantic_scorer.encode(gt_list, convert_to_tensor=True, device='cpu')
            cosine_scores = util.cos_sim(pred_emb, gt_emb)

            pred_max = cosine_scores.max(dim=1).values
            tp_precision = (pred_max >= SIMILARITY_THRESHOLD).sum().item()
            bigearth_semantic_precision.append(tp_precision / len(pred_list))

            gt_max = cosine_scores.max(dim=0).values
            tp_recall = (gt_max >= SIMILARITY_THRESHOLD).sum().item()
            bigearth_semantic_recall.append(tp_recall / len(gt_list))

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        continue

# ==============================================================================
# 5. FINAL REPORT
# ==============================================================================
print("\n" + "="*50)
print("MULTI-TASK VLM FINAL BENCHMARK SCORE")
print("="*50)

if vrsbench_scores:
    avg_rouge = sum(vrsbench_scores) / len(vrsbench_scores)
    print(f"[VRSBench - Visual Question Answering (Eval Split)]")
    print(f"Total Test Samples : {len(vrsbench_scores)}")
    print(f"Average ROUGE-L    : {avg_rouge:.4f}\n")

if bigearth_semantic_precision:
    avg_prec = sum(bigearth_semantic_precision) / len(bigearth_semantic_precision)
    avg_rec = sum(bigearth_semantic_recall) / len(bigearth_semantic_recall)
    f1 = 2 * (avg_prec * avg_rec) / (avg_prec + avg_rec) if (avg_prec + avg_rec) > 0 else 0.0

    print(f"[BigEarthNet - Semantic Classification (Test Split | Threshold > {SIMILARITY_THRESHOLD})]")
    print(f"Total Test Samples : {len(bigearth_semantic_precision)}")
    print(f"Semantic Precision : {avg_prec:.4f}")
    print(f"Semantic Recall    : {avg_rec:.4f}")
    print(f"Semantic F1-Score  : {f1:.4f}")
print("="*50)

# ==============================================================================
# 6. SINGLE-IMAGE SPOT TEST
# ==============================================================================
print("\nExecuting Single Image Spot-Test...")
spot_sample = random.choice(test_data)
if spot_sample['dataset'] == 'VRSBench':
    spot_img = Image.open(spot_sample['image_path']).convert("RGB")
else:
    spot_img = spot_sample['image_obj']

prompt = f"USER: <image>\n{spot_sample['question']}\nASSISTANT:"
inputs = processor(text=prompt, images=spot_img, return_tensors="pt").to("cuda", torch.float16)

with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=75, do_sample=False)
prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].split("ASSISTANT:")[-1].strip()

print("\n[SPOT TEST RESULT]")
print(f"Dataset      : {spot_sample['dataset']}")
print(f"Question     : {spot_sample['question']}")
print(f"Ground Truth : {spot_sample['answer']}")
print(f"Model Output : {prediction}")